In [1]:
import socket
import threading
import time

In [2]:
common_ports = {
    21: "FTP - File Transfer Protocol",
    22: "SSH - Secure Shell",
    25: "SMTP - Email Transfer",
    53: "DNS - Domain Name System",
    80: "HTTP - Web Traffic",
    110: "POP3 - Email Retrieval",
    143: "IMAP - Email Retrieval",
    443: "HTTPS - Secure Web Traffic",
    3306: "MySQL - Database",
    5432: "PostgreSQL - Database",
    8080: "HTTP Alternate - Web Service"
}

print("Loaded", len(common_ports), "common TCP ports.")

Loaded 11 common TCP ports.


In [4]:
test_port = 8081

def start_test_server():
    server = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    server.setsockopt(
        socket.SOL_SOCKET,
        socket.SO_REUSEADDR,
        1
    )

    try:
        server.bind(("127.0.0.1", test_port))
        server.listen(5)
        server.settimeout(0.5)

        print(f"Test service listening on port {test_port}")

        end_time = time.time() + 60

        while time.time() < end_time:
            try:
                connection, address = server.accept()
                connection.close()
            except socket.timeout:
                continue

    except OSError as error:
        print("Could not start test service:", error)

    finally:
        server.close()


server_thread = threading.Thread(
    target=start_test_server,
    daemon=True
)

server_thread.start()
time.sleep(1)

print("Local test service is ready.")

Test service listening on port 8081
Local test service is ready.


In [5]:
def scan_port(host, port, timeout=1):
    try:
        with socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        ) as sock:

            sock.settimeout(timeout)
            result = sock.connect_ex((host, port))

            if result == 0:
                return "Open"

            return "Closed"

    except socket.gaierror:
        return "Hostname could not be resolved"

    except socket.timeout:
        return "Filtered or timed out"

    except OSError as error:
        return f"Network error: {error}"

In [6]:
target_host = "127.0.0.1"

print("Target:", target_host)
print("Starting TCP port scan...")
print("-" * 55)

scan_results = {}

for port, service in common_ports.items():

    status = scan_port(target_host, port)
    scan_results[port] = status

    print(
        f"Port {port:5} | "
        f"{status:30} | "
        f"{service}"
    )

print("-" * 55)
print("Scan completed.")

Target: 127.0.0.1
Starting TCP port scan...
-------------------------------------------------------
Port    21 | Closed                         | FTP - File Transfer Protocol
Port    22 | Closed                         | SSH - Secure Shell
Port    25 | Closed                         | SMTP - Email Transfer
Port    53 | Closed                         | DNS - Domain Name System
Port    80 | Closed                         | HTTP - Web Traffic
Port   110 | Closed                         | POP3 - Email Retrieval
Port   143 | Closed                         | IMAP - Email Retrieval
Port   443 | Closed                         | HTTPS - Secure Web Traffic
Port  3306 | Closed                         | MySQL - Database
Port  5432 | Closed                         | PostgreSQL - Database
Port  8080 | Open                           | HTTP Alternate - Web Service
-------------------------------------------------------
Scan completed.


In [7]:
open_ports = [
    port for port, status in scan_results.items()
    if status == "Open"
]

closed_ports = [
    port for port, status in scan_results.items()
    if status == "Closed"
]

unconfirmed_ports = [
    port for port, status in scan_results.items()
    if status not in ("Open", "Closed")
]

print("=" * 55)
print("             PORT SCAN SUMMARY")
print("=" * 55)

print("Target:", target_host)
print("Total ports checked:", len(scan_results))
print("Open ports:", len(open_ports))
print("Closed ports:", len(closed_ports))
print("Unconfirmed ports:", len(unconfirmed_ports))

print("\nOpen ports and typical services:")

if open_ports:
    for port in open_ports:
        print(f"- {port}: {common_ports[port]}")
else:
    print("No open ports detected.")

print("\nSecurity interpretation:")
if open_ports:
    print(
        "Review open ports to confirm that their services "
        "are necessary, authorized, and appropriately secured."
    )
else:
    print(
        "No open ports were detected in this scan. "
        "This does not prove that the host has no exposed services."
    )

print("=" * 55)

             PORT SCAN SUMMARY
Target: 127.0.0.1
Total ports checked: 11
Open ports: 1
Closed ports: 10
Unconfirmed ports: 0

Open ports and typical services:
- 8080: HTTP Alternate - Web Service

Security interpretation:
Review open ports to confirm that their services are necessary, authorized, and appropriately secured.
